# Lab 3 — Simple forecasting models

**Goal:** implement **benchmark** methods — naive, seasonal naive, drift, moving-average, and **linear regression** on time + harmonics — and compare them with a proper **train/test** split on `data/synthetic_series.csv`.

**Prerequisite:** `generate_data.ipynb`.

Theory: `time_series_forecasting.md` (random walk ↔ naive, seasonal differencing ↔ seasonal naive, regression assumptions, train vs test evaluation).

## 1. Why benchmarks matter

Complex models should **beat** simple baselines on out-of-sample metrics. If they do not, you may have **overfitting**, a **bug**, or a **mis-specified** horizon.

**One-step vs multi-step:** on the test set, **rolling one-step** forecasts (re-fit or update with each new observation) are often easier than a single **long multi-step** run from one origin — errors compound in the latter. This lab uses **one-step** test evaluation for comparability.

## 2. Classic formulas (horizon \(h=1\) from origin at \(t\))

- **Naive:** \(\hat{y}_{t+1} = y_t\).
- **Seasonal naive:** \(\hat{y}_{t+1} = y_{t+1-m}\) where \(m\) is seasonal period (requires \(t+1-m \ge 1\)).
- **Drift:** \(\hat{y}_{t+1} = y_t + \frac{y_t - y_1}{t-1}\) (extrapolate average slope from start).
- **Moving average:** \(\hat{y}_{t+1} = \frac{1}{k}\sum_{j=1}^{k} y_{t+1-j}\) (mean of last \(k\) **past** values).

## 3. Metrics

- **MAE** — mean absolute error, same units as \(y\).
- **RMSE** — penalises large errors.
- **MAPE** — relative error; unstable if \(y\approx 0\) (our synthetic \(y>0\)).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = Path("data/synthetic_series.csv")
META_PATH = Path("data/synthetic_series_meta.json")
raw = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date")
df = raw.set_index("date").asfreq("D")
if META_PATH.exists():
    with open(META_PATH, encoding="utf-8") as f:
        m = int(json.load(f).get("seasonal_period", 7))
else:
    m = 7

y = df["y"].astype(float)
TEST_H = 120
train, test = y.iloc[:-TEST_H], y.iloc[-TEST_H:]
print(len(train), "train", len(test), "test", "m =", m)

In [ ]:
def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-9))) * 100)


def eval_forecast(name: str, y_true: pd.Series, y_pred: pd.Series) -> dict:
    yt = y_true.values
    yp = y_pred.values
    return {
        "model": name,
        "MAE": mean_absolute_error(yt, yp),
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
        "MAPE": mape(yt, yp),
    }


def rolling_one_step_naive(train: pd.Series, test: pd.Series, mode: str, k_ma: int = 7) -> pd.Series:
    """Expand history with observed test values; one-step predictions for each test index."""
    hist = train.copy()
    preds = []
    idx = []
    for t in test.index:
        if mode == "naive":
            p = hist.iloc[-1]
        elif mode == "seasonal":
            p = hist.iloc[-m] if len(hist) >= m else np.nan
        elif mode == "drift":
            if len(hist) < 2:
                p = hist.iloc[-1]
            else:
                p = hist.iloc[-1] + (hist.iloc[-1] - hist.iloc[0]) / (len(hist) - 1)
        elif mode == "ma":
            p = hist.iloc[-k_ma:].mean()
        else:
            raise ValueError(mode)
        preds.append(p)
        idx.append(t)
        hist = pd.concat([hist, test.loc[[t]]])
    return pd.Series(preds, index=idx)


results = []
forecasts = {}
for mode, label in [
    ("naive", "Naive"),
    ("seasonal", "Seasonal naive"),
    ("drift", "Drift"),
    ("ma", f"MA(k={7})"),
]:
    pred = rolling_one_step_naive(train, test, mode, k_ma=7)
    forecasts[label] = pred
    results.append(eval_forecast(label, test, pred))

pd.DataFrame(results)

In [ ]:
# Linear regression: y ~ time + Fourier(m) + optional exog — fit on train only
def design_matrix(series: pd.Series) -> np.ndarray:
    n = len(series)
    t = np.arange(n, dtype=float).reshape(-1, 1)
    K = 3
    fcols = []
    for j in range(1, K + 1):
        fcols.append(np.sin(2 * np.pi * j * t.flatten() / m).reshape(-1, 1))
        fcols.append(np.cos(2 * np.pi * j * t.flatten() / m).reshape(-1, 1))
    X = np.hstack([np.ones((n, 1)), t] + fcols)
    if {"temp", "promo"}.issubset(df.columns):
        ex = df.loc[series.index, ["temp", "promo"]].values
        X = np.hstack([X, ex])
    return X


X_tr = design_matrix(train)
lr = LinearRegression(fit_intercept=False)
lr.fit(X_tr, train.values)

hist_len = len(train)
pred_reg = []
hist = train.copy()
for t in test.index:
    # one-step: predict next from features at end of hist
    n = len(hist)
    tvec = np.array([[n]])
    row = [1.0, float(n)]
    for j in range(1, 4):
        row.append(np.sin(2 * np.pi * j * n / m))
        row.append(np.cos(2 * np.pi * j * n / m))
    if {"temp", "promo"}.issubset(df.columns):
        row.extend(df.loc[t, ["temp", "promo"]].astype(float).tolist())
    x = np.array(row).reshape(1, -1)
    pred_reg.append(float(lr.predict(x)))
    hist = pd.concat([hist, test.loc[[t]]])

s_reg = pd.Series(pred_reg, index=test.index)
forecasts["Linear regression"] = s_reg
results.append(eval_forecast("Linear regression", test, s_reg))
pd.DataFrame(results)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test.index, test.values, "k", lw=1, label="Actual")
for name, col in [("Naive", "C0"), ("Seasonal naive", "C1"), ("Linear regression", "C2")]:
    ax.plot(test.index, forecasts[name].values, lw=0.9, alpha=0.85, label=name, color=col)
ax.legend(loc="upper left", fontsize=8)
ax.set_title("Test set: one-step rolling forecasts")
plt.tight_layout()
plt.show()

## Interview checklist

1. State **horizon** and **origin** when you report accuracy.
2. **Naive** is hard to beat on **stationary** or **random-walk-like** series after differencing.
3. Regression errors should be **uncorrelated** if the linear specification is adequate — check residual ACF in practice.

**Next:** Lab 4 — ARIMA / SARIMA / ARIMAX.